# 22 AG baselines: verification and the baseline band

**Purpose.** To show how much each ledger entry depends on which no-mandate run serves as the
baseline, this notebook carries out protocol §7 (signed 2026-10-28; amendments A9–A12).

1. **Verification.** Each of the 19 no-mandate runs (11 in the reference market, 8 under the emissions
   constraint; paper 1's `v1f_common.py`) must build no SMR in any solve year, and its new
   large-reactor MW in 2026–2050 must equal the exogenous units in ReEDS's generator database
   (Vogtle 4 and three restarts; amendment A9). A run that fails leaves the band, and the reason is recorded.
2. **Level spread.** Prices, payments, system cost, gas price and emissions of the 19 runs, by year.
3. **Baseline band (reference market).** The stage-1 ledger of the six P50 runs (notebook 21) is
   rebuilt against each verified reference baseline. The band is the range of every actor entry and
   of both resident views across the verified baselines without a nuclear ITC. `fbR_itc30` pays a 30%
   ITC to the exogenous nuclear units, so it is reported beside the band as a sensitivity (amendment A13). The cap-regime band needs the allowance entries of stage 2c
   (protocol §4.8) and is built with stage 2c.

**What it writes** (`results/exports/22/`): `ag_baseline_check.csv`, `ag_baseline_levels.csv`,
`ag_band_actor_pv.csv`, `ag_band_views.csv`, `ag_band_closure.csv`. It opens no ReEDS outputs file and
states no hypothesis.


In [1]:
import sys
from pathlib import Path
_p = Path.cwd().resolve()
while not (_p / "nlpaper").is_dir():           # find z-ethan/ from wherever the kernel starts
    _p = _p.parent
sys.path.insert(0, str(_p))
import numpy as np
import pandas as pd

import nlpaper
from nlpaper import paths, plotstyle as ps
from aglib import ledger as L
nlpaper.setup(threads=4)
pd.set_option("display.width", 220)
pd.set_option("display.max_rows", 300)
pd.set_option("display.max_columns", 30)
OUT = paths.exports("22")
TO25 = ps.to2025(2004)
TARGETS = ["eia", "aj", "iaea", "mck", "cop28", "eo"]
REF_BASES = [("itcfb", "fbR_none"), ("itcfb", "fbR_itc30"),
             ("batch2", "bx_aj_p75_f50"), ("batch2", "bx_cop28_p50_f50"), ("batch2", "bx_cop28_p75_f50"),
             ("batch2", "bx_eia_p75_f50"), ("batch2", "bx_eo_p75_f50"), ("batch2", "bx_iaea_p50_f50"),
             ("batch2", "bx_iaea_p75_f50"), ("batch2", "bx_mck_p75_f50"), ("ratedesign2", "ax_eia_p95_c60")]
CAP_BASES = [("batch3", c) for c in ["bx_aj_p75_f50_co2cap", "bx_cop28_p50_f50_co2cap", "bx_cop28_p75_f50_co2cap",
             "bx_eia_p75_f50_co2cap", "bx_eo_p75_f50_co2cap", "bx_iaea_p50_f50_co2cap",
             "bx_iaea_p75_f50_co2cap", "bx_mck_p75_f50_co2cap"]]
PRIMARY = ("itcfb", "fbR_none")
ITC_SENS = ("itcfb", "fbR_itc30")      # pays an ITC to the exogenous nuclear units: reported beside the band (amendment A13)
rd = lambda b, k: pd.read_parquet(L.EXTRACTS_AG / b / f"{k}.parquet")
print(f"{len(REF_BASES)} reference + {len(CAP_BASES)} cap baselines; exports to {paths.rel(OUT)}")

nlpaper setup: python 3.12.3 | numpy 1.26.4 | pandas 2.2.2 | matplotlib 3.8.4 | threads 4
determinism: seeds are explicit numpy Generators in each cell; results do not depend on PYTHONHASHSEED.
11 reference + 8 cap baselines; exports to z-ethan\results\exports\22


## 1. Verification (protocol §7 step 1, amendment A9)

In [2]:
gdb = pd.read_csv(paths.REPO / "inputs" / "capacity_exogenous" / "ReEDS_generator_database_final_EIA-NEMS.csv",
                  usecols=["tech", "StartYear", "summer_power_capacity_MW"], low_memory=False)
EXO_MW = float(gdb[(gdb["tech"] == "nuclear") & (gdb["StartYear"] > 2023)]["summer_power_capacity_MW"].sum())
rows = []
for regime, bases in (("reference", REF_BASES), ("cap", CAP_BASES)):
    for b, c in bases:
        cn = rd(b, "cap_new_ivrt"); cn = cn[(cn["case"] == c) & (cn["t"].astype(int) >= 2026)]
        cap = rd(b, "cap_nat"); cap = cap[(cap["case"] == c) & (cap["i"] == "nuclear-smr")]
        dl = rd(b, "dropped_load"); dl = dl[(dl["case"] == c) & (dl["t"].astype(int) >= 2026)]
        yrs = sorted(rd(b, "pvf_onm").query("case == @c")["t"].astype(int))
        new_lr = float(cn.loc[cn["i"] == "nuclear", "Value"].sum())
        rows.append(dict(regime=regime, batch=b, case=c, last_year=max(yrs),
                         smr_new_MW=float(cn.loc[cn["i"] == "nuclear-smr", "Value"].sum()),
                         smr_fleet_MW_max=float(cap["Value"].max()) if len(cap) else 0.0,
                         large_new_MW=new_lr, exogenous_MW=EXO_MW,
                         dropped_load_MWh=float(dl["Value"].abs().sum()) if len(dl) else 0.0))
CHK = pd.DataFrame(rows)
CHK["verified"] = ((CHK["smr_new_MW"] < 1e-6) & (CHK["smr_fleet_MW_max"] < 1e-6)
                   & ((CHK["large_new_MW"] - CHK["exogenous_MW"]).abs() < 0.5)
                   & (CHK["dropped_load_MWh"] < 1e-6) & (CHK["last_year"] == 2050))
CHK.to_csv(OUT / "ag_baseline_check.csv", index=False)
print(CHK.round(1).to_string(index=False))
print("\nverified:", int(CHK["verified"].sum()), "of", len(CHK))
VERIFIED_REF = [(b, c) for b, c in REF_BASES if CHK.set_index("case").loc[c, "verified"]]

   regime       batch                    case  last_year  smr_new_MW  smr_fleet_MW_max  large_new_MW  exogenous_MW  dropped_load_MWh  verified
reference       itcfb                fbR_none       2050         0.0               0.0        3286.7        3286.7               0.0      True
reference       itcfb               fbR_itc30       2050         0.0               0.0        3286.7        3286.7               0.0      True
reference      batch2           bx_aj_p75_f50       2050         0.0               0.0        3286.7        3286.7               0.0      True
reference      batch2        bx_cop28_p50_f50       2050         0.0               0.0        3286.7        3286.7               0.0      True
reference      batch2        bx_cop28_p75_f50       2050         0.0               0.0        3286.7        3286.7               0.0      True
reference      batch2          bx_eia_p75_f50       2050         0.0               0.0        3286.7        3286.7               0.0      True

## 2. Level spread of the 19 baselines

Per run and solve year (2004 $): the average energy price P_E / Q_E, capacity and RPS payments,
system cost (all rows; lumps in their solve year), the national gas price and process CO2. The table
gives the range across runs within each regime, in 2025, 2035 and 2050 values.

In [3]:
rows = []
for regime, bases in (("reference", REF_BASES), ("cap", CAP_BASES)):
    for b, c in bases:
        der = pd.read_parquet(L.EXTRACTS_AG / b / "derived_ledger.parquet").query("case == @c")
        der = der.pivot_table(index="t", columns="item", values="Value")
        sc = rd(b, "systemcost").query("case == @c").groupby("t")["Value"].sum()
        gp = rd(b, "repgasprice_nat").query("case == @c").set_index("t")["Value"]
        em = rd(b, "emit_nat").query("case == @c and eall == 'CO2' and etype == 'process'").set_index("t")["Value"]
        d = pd.DataFrame({"energy_price_per_MWh": der["P_E"] / der["Q_E"], "P_E_B": der["P_E"] / 1e9,
                          "P_C_B": der["P_C"] / 1e9, "P_R_B": der["P_R"] / 1e9, "systemcost_B": sc / 1e9,
                          "gas_price_per_MMBtu": gp, "CO2_process_Mt": em / 1e6})
        d.index = d.index.astype(int)
        rows.append(d[d.index >= 2026].assign(regime=regime, case=c).reset_index(names="t"))
LEV = pd.concat(rows, ignore_index=True)
LEV.to_csv(OUT / "ag_baseline_levels.csv", index=False)
cols = ["energy_price_per_MWh", "P_E_B", "P_C_B", "P_R_B", "systemcost_B", "gas_price_per_MMBtu", "CO2_process_Mt"]
for regime in ("reference", "cap"):
    x = LEV[(LEV["regime"] == regime) & LEV["t"].isin([2035, 2050])]
    print(f"\n{regime} (2004 $):")
    print(x.groupby("t")[cols].agg(["min", "max"]).round(2).T.to_string())


reference (2004 $):
t                               2035        2050
energy_price_per_MWh min   23.010000   21.730000
                     max   23.010000   21.730000
P_E_B                min  126.209999  148.809998
                     max  126.220001  148.820007
P_C_B                min   84.250000  103.199997
                     max   84.250000  103.209999
P_R_B                min    2.750000   10.250000
                     max    2.750000   10.260000
systemcost_B         min  177.089996  273.890015
                     max  177.110001  273.940002
gas_price_per_MMBtu  min    2.860000    2.540000
                     max    2.860000    2.540000
CO2_process_Mt       min  685.869995  848.580017
                     max  685.890015  848.599976

cap (2004 $):
t                               2035        2050
energy_price_per_MWh min   23.010000   31.900000
                     max   23.010000   31.900000
P_E_B                min  126.190002  218.490005
                     max  126.190

## 3. The baseline band (reference market)

Each of the six P50 runs is paired with each verified reference baseline. The band of an entry is
its minimum and maximum across the band baselines, beside its value against the primary baseline `fbR_none`
and against the ITC sensitivity `fbR_itc30`
(notebook 21). Display configuration as in notebook 21 §6 (funding T, lump support); every
combination is in the exports.

In [4]:
EXT = L.load_external()
CRF = L.crf_after_2026()
AP, VW, CL3 = [], [], []
for t in TARGETS:
    run = ("smr100", f"smr100_{t}_p50")
    for base in VERIFIED_REF:
        o = L.pair(run, base, EXT, CRF)
        tag = dict(target=t, baseline=base[1], primary=(base == PRIMARY), in_band=(base != ITC_SENS))
        AP.append(L.actor_pvs(o, True).assign(**tag))
        v = L.views(o, True)
        VW.append(v.assign(**tag))
        ck = o["checks"].set_index(["who", "t"])
        d = lambda c: ck.loc["run", c] - ck.loc["base", c]
        named = (d("C1_residual") - d("C1_vsc_rent")) + (d("C2_residual") - d("C2_vsc_rent")) + d("RPS_residual") - d("RPS_assigned")
        for (h, rk), g in v.groupby(["horizon", "rate"]):
            CL3.append(dict(**tag, horizon=h, rate=rk, C3=g["C3_residual"].iloc[0],
                            unexplained=g["C3_residual"].iloc[0] - L.pv(L.flow(named, h), h, L.RATES[rk])))
    print(f"{t}: {len(VERIFIED_REF)} baselines done")
AP = pd.concat(AP, ignore_index=True); VW = pd.concat(VW, ignore_index=True); CL3 = pd.DataFrame(CL3)
AP.to_csv(OUT / "ag_band_actor_pv.csv", index=False)
VW.to_csv(OUT / "ag_band_views.csv", index=False)
CL3.to_csv(OUT / "ag_band_closure.csv", index=False)
print("C3 max |unexplained| over all pairs (2025 $B):", float(CL3["unexplained"].abs().max()))

eia: 11 baselines done


aj: 11 baselines done


iaea: 11 baselines done


mck: 11 baselines done


cop28: 11 baselines done


eo: 11 baselines done


C3 max |unexplained| over all pairs (2025 $B): 0.0004742935402486381


**Actor totals: value against the primary baseline and the band across baselines** (horizon A,
2.0%, funding T, lump support; 2025 $B, PV to 2026). The environment is shown for one carbon basis and
one health variant.

In [5]:
FREE = lambda d: d[d["allowance_state"].isin(["free", "none"]) & d["allowance_national"].isin(["free", "none"])]
sel = FREE(AP[(AP["funding"] == "T") & (AP["support_form"] == "lump") & (AP["horizon"] == "A")])
money = sel[sel["actor"] != "environment"].groupby(["target", "baseline", "primary", "in_band", "actor"])["PV_2.0%"].sum().reset_index()
env = sel[sel["entry"].isin(["GHG: global EPA2023 2.0%", "health: AP2 ACS"])].assign(actor=lambda d: "environment, " + d["entry"])
env = env.groupby(["target", "baseline", "primary", "in_band", "actor"])["PV_2.0%"].sum().reset_index()
T = pd.concat([money, env])
prim = T[T["primary"]].set_index(["actor", "target"])["PV_2.0%"]
band = T[T["in_band"]].groupby(["actor", "target"])["PV_2.0%"].agg(["min", "max"])
band["primary"] = prim
band["width"] = band["max"] - band["min"]
band["vs_itc30"] = T[T["baseline"] == ITC_SENS[1]].set_index(["actor", "target"])["PV_2.0%"]
for t in ("eia", "eo"):
    print(f"\n{t}:")
    print(band.xs(t, level="target")[["primary", "min", "max", "width", "vs_itc30"]].round(1).to_string())


eia:
                                       primary   min   max  width  vs_itc30
actor                                                                      
electricity consumers                     32.7  32.7  32.7    0.0      32.7
environment, GHG: global EPA2023 2.0%     43.4  43.4  43.4    0.0      43.4
environment, health: AP2 ACS               0.4   0.4   0.4    0.0       0.4
foreign buyers                             7.8   7.8   7.8    0.0       7.8
foreign sellers: Canadian imports         -1.2  -1.3  -1.2    0.1      -1.2
gas producers                            -26.9 -26.9 -26.9    0.0     -26.9
network                                    0.6   0.6   0.7    0.0       0.6
non-power gas consumers                   10.8  10.8  10.8    0.0      10.8
owners: H2 and DAC production             -0.0  -0.0  -0.0    0.0      -0.0
owners: SMR (new)                          9.1   9.1   9.1    0.0       9.1
owners: coal                              -0.5  -0.7  -0.4    0.2      -0.5
owners

In [6]:
ref = VW[(VW["funding"] == "T") & (VW["support_form"] == "lump") & (VW["health_model"] == "AP2 ACS")
         & (VW["mcpf"] == 1.0) & (VW["phi"] == 0.0) & VW["allowance_state"].isin(["free", "none"])]
for h in ("A", "B"):
    x = ref[(ref["horizon"] == h) & ref["scc_basis"].isin(["global EPA2023 2.0%", "domestic EPA2019 3%"])]
    g = x[x["in_band"]].groupby(["scc_basis", "target"])
    s30 = x[x["baseline"] == ITC_SENS[1]].set_index(["scc_basis", "target"])
    out = pd.DataFrame({"view1_primary": x[x["primary"]].set_index(["scc_basis", "target"])["view1"],
                        "view1_min": g["view1"].min(), "view1_max": g["view1"].max(),
                        "view2_primary": x[x["primary"]].set_index(["scc_basis", "target"])["view2"],
                        "view2_min": g["view2"].min(), "view2_max": g["view2"].max(),
                        "view1_vs_itc30": s30["view1"], "view2_vs_itc30": s30["view2"]})
    print(f"\nhorizon {h} (reference view variant):")
    print(out.round(1).to_string())


horizon A (reference view variant):
                            view1_primary  view1_min  view1_max  view2_primary  view2_min  view2_max  view1_vs_itc30  view2_vs_itc30
scc_basis           target                                                                                                          
domestic EPA2019 3% aj               20.4       20.4       20.4          -34.8      -34.8      -34.3            30.6           -34.7
                    cop28           135.8      135.8      135.8         -257.7     -257.7     -257.1           145.9          -257.5
                    eia               1.0        0.9        1.0          -38.8      -38.8      -38.2            11.1           -38.6
                    eo              274.6      274.6      274.6         -332.7     -332.7     -332.2           284.8          -332.5
                    iaea             13.4       13.4       13.4         -133.2     -133.2     -132.6            23.6          -133.0
                    mck         

## 4. What reads these exports

The band joins the stage-2 ledger (Phase 4) as an uncertainty range on every entry. Observations
go to `docs/observations.md` by hand (protocol §10).